# BipedalWalker-v3 Baseline: SAC + gSDE, saved to Google Drive, resumable

Same SAC + gSDE + `VecNormalize` + `SubprocVecEnv` setup as before, tuned with
the RL-Zoo-style hyperparameters (`gamma=0.98`, `train_freq=64`,
`gradient_steps=64`, `net_arch=[400, 300]`, `log_std_init=-3`) that address
BipedalWalker's "stand rigid and don't fall" local optimum.

**What's new: everything persists to Google Drive, and training can resume
from any checkpoint.** Colab's local disk (`/tmp/...`) disappears the moment
the runtime disconnects — this notebook instead:

1. Mounts your Drive and saves periodic checkpoints there during training
   (model weights, `VecNormalize` running stats, and the replay buffer —
   all three are needed to resume SAC properly, not just the model).
2. Lets you set `CHECKPOINT_TO_RESUME` to a saved checkpoint's step count and
   continue training from exactly where you left off, instead of starting
   over from step 0.

Run the "start fresh" path once, then any time the runtime disconnects, come
back, remount Drive, list what's in your checkpoint folder, and set
`CHECKPOINT_TO_RESUME` to pick up again.

In [ ]:
!pip install -q "stable-baselines3[extra]" "gymnasium[box2d]" 

In [ ]:
import os
import numpy as np
import gymnasium as gym
import matplotlib.pyplot as plt

from stable_baselines3 import SAC
from stable_baselines3.common.env_util import make_vec_env
from stable_baselines3.common.vec_env import VecNormalize, SubprocVecEnv
from stable_baselines3.common.evaluation import evaluate_policy
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.results_plotter import load_results, ts2xy
from stable_baselines3.common.callbacks import CheckpointCallback

## 1. Mount Google Drive

Everything this notebook saves — checkpoints, the final model, logs — goes
under `DRIVE_DIR`. Change the path if you want it somewhere other than the
default `MyDrive/bipedalwalker_sac/` folder.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_DIR = "/content/drive/MyDrive/bipedalwalker_sac"
CHECKPOINT_DIR = os.path.join(DRIVE_DIR, "checkpoints")
LOG_DIR = os.path.join(DRIVE_DIR, "logs")

os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(LOG_DIR, exist_ok=True)
print(f"Saving everything under: {DRIVE_DIR}")

## 2. Resume settings

- Leave `CHECKPOINT_TO_RESUME = None` to start a fresh training run.
- To resume, list what's in `CHECKPOINT_DIR` (next cell), find the step count
  you want to resume from, and set `CHECKPOINT_TO_RESUME = <that number>`.

Checkpoints are named `sac_bipedalwalker_<steps>_steps.zip`, with matching
`sac_bipedalwalker_vecnormalize_<steps>_steps.pkl` and
`sac_bipedalwalker_replay_buffer_<steps>_steps.pkl` files alongside — all
three get loaded together when resuming, since SAC needs its replay buffer
and the observation normalization stats to continue meaningfully, not just
the network weights.

In [ ]:
# List available checkpoints
if os.path.exists(CHECKPOINT_DIR):
    checkpoints = sorted(os.listdir(CHECKPOINT_DIR))
    if checkpoints:
        print("Available checkpoint files:")
        for f in checkpoints:
            print(" ", f)
    else:
        print("No checkpoints yet — this will be a fresh run.")
else:
    print("No checkpoints yet — this will be a fresh run.")

In [ ]:
CHECKPOINT_TO_RESUME = None  # e.g. 200000 to resume from sac_bipedalwalker_200000_steps.zip
ADDITIONAL_TIMESTEPS = 400_000  # how many MORE steps to train, whether starting fresh or resuming

## 3. Baseline reference: random policy

Skip re-running this if you're resuming and already have this number from a
prior session.

In [ ]:
def run_random_baseline(n_episodes=5):
    env = gym.make("BipedalWalker-v3")
    rewards = []
    for ep in range(n_episodes):
        obs, info = env.reset(seed=ep)
        total_reward = 0
        terminated = truncated = False
        while not (terminated or truncated):
            action = env.action_space.sample()
            obs, reward, terminated, truncated, info = env.step(action)
            total_reward += reward
        rewards.append(total_reward)
    env.close()
    return rewards

random_rewards = run_random_baseline()
print(f"Random policy rewards: {random_rewards}")
print(f"Mean: {np.mean(random_rewards):.1f}")

## 4. Set up the (parallel, normalized) training environment

Same `SubprocVecEnv` + `VecNormalize` setup as before. If resuming, the
`VecNormalize` wrapper is loaded from the saved stats instead of created
fresh — using the wrong (or no) normalization stats with a resumed model
will make it behave badly, since it was trained on normalized observations.

In [ ]:
N_ENVS = 8
vec_env = make_vec_env(
    "BipedalWalker-v3",
    n_envs=N_ENVS,
    monitor_dir=LOG_DIR,
    vec_env_cls=SubprocVecEnv,
)

if CHECKPOINT_TO_RESUME is not None:
    vecnorm_path = os.path.join(CHECKPOINT_DIR, f"sac_bipedalwalker_vecnormalize_{CHECKPOINT_TO_RESUME}_steps.pkl")
    vec_env = VecNormalize.load(vecnorm_path, vec_env)
    print(f"Loaded VecNormalize stats from: {vecnorm_path}")
else:
    vec_env = VecNormalize(vec_env, norm_obs=True, norm_reward=True, clip_reward=10.0)
    print("Fresh VecNormalize wrapper.")

## 5. Create or load the model

Fresh run: builds a new SAC model with the tuned hyperparameters (`gamma=0.98`,
`train_freq=64`, `gradient_steps=64`, `net_arch=[400, 300]`, `log_std_init=-3`
— the RL-Zoo-style settings that address the "stand rigid" local optimum).

Resume: loads the saved model weights *and* the saved replay buffer, so SAC's
off-policy learning continues with the experience it had already collected
rather than starting the buffer over empty.

In [ ]:
if CHECKPOINT_TO_RESUME is not None:
    model_path = os.path.join(CHECKPOINT_DIR, f"sac_bipedalwalker_{CHECKPOINT_TO_RESUME}_steps.zip")
    model = SAC.load(model_path, env=vec_env, device="cpu")

    replay_buffer_path = os.path.join(CHECKPOINT_DIR, f"sac_bipedalwalker_replay_buffer_{CHECKPOINT_TO_RESUME}_steps.pkl")
    if os.path.exists(replay_buffer_path):
        model.load_replay_buffer(replay_buffer_path)
        print(f"Loaded replay buffer from: {replay_buffer_path}")
    else:
        print("No saved replay buffer found — resuming with an empty buffer (learning will be less stable at first).")

    print(f"Resumed model from: {model_path} (was at {model.num_timesteps} timesteps)")
else:
    model = SAC(
        "MlpPolicy",
        vec_env,
        verbose=1,
        device="cpu",
        learning_rate=7.3e-4,
        buffer_size=300_000,
        learning_starts=10_000,
        batch_size=256,
        tau=0.005,
        gamma=0.98,
        train_freq=64,
        gradient_steps=64,
        use_sde=True,
        sde_sample_freq=4,
        policy_kwargs=dict(net_arch=[400, 300], log_std_init=-3),
    )
    print("Created a fresh model.")

## 6. Train, checkpointing to Drive as it goes

`CheckpointCallback` saves the model, `VecNormalize` stats, and replay buffer
together every `save_freq` steps, so a mid-training disconnect only costs you
back to the last checkpoint — not the whole run.

`save_freq` is counted in steps-per-environment internally when using a
vectorized env, so it's divided by `N_ENVS` to get roughly the requested
total-timestep interval between saves.

In [ ]:
SAVE_EVERY_TIMESTEPS = 20_000

checkpoint_callback = CheckpointCallback(
    save_freq=max(SAVE_EVERY_TIMESTEPS // N_ENVS, 1),
    save_path=CHECKPOINT_DIR,
    name_prefix="sac_bipedalwalker",
    save_replay_buffer=True,
    save_vecnormalize=True,
)

model.learn(
    total_timesteps=ADDITIONAL_TIMESTEPS,
    reset_num_timesteps=False,  # keep the timestep counter continuous across resumes
    callback=checkpoint_callback,
    progress_bar=True,
)

# Also save a "latest" copy under stable filenames for convenience, alongside the
# step-numbered checkpoints from the callback above.
model.save(os.path.join(DRIVE_DIR, "sac_bipedalwalker_latest"))
vec_env.save(os.path.join(DRIVE_DIR, "sac_bipedalwalker_vecnormalize_latest.pkl"))
model.save_replay_buffer(os.path.join(DRIVE_DIR, "sac_bipedalwalker_replay_buffer_latest.pkl"))
print(f"Saved latest model/stats/buffer to: {DRIVE_DIR}")
print(f"Total timesteps so far: {model.num_timesteps}")

## 7. Plot the learning curve

Reads all `Monitor` CSVs in `LOG_DIR`, which now include logs from every
session (fresh + any resumes), since they all wrote into the same Drive
folder.

In [ ]:
def plot_results(log_dir, title="Learning Curve"):
    x, y = ts2xy(load_results(log_dir), "timesteps")
    window = 20
    if len(y) >= window:
        y_smooth = np.convolve(y, np.ones(window) / window, mode="valid")
        x_smooth = x[window - 1:]
    else:
        y_smooth, x_smooth = y, x

    plt.figure(figsize=(8, 5))
    plt.plot(x, y, alpha=0.3, label="episode reward")
    plt.plot(x_smooth, y_smooth, label=f"rolling mean ({window} episodes)")
    plt.axhline(300, color="green", linestyle="--", label="'solved' threshold (300)")
    plt.axhline(np.mean(random_rewards), color="red", linestyle="--", label="random baseline")
    plt.xlabel("timesteps")
    plt.ylabel("episode reward")
    plt.title(title)
    plt.legend()
    plt.tight_layout()
    plt.show()

plot_results(LOG_DIR, "SAC + gSDE on BipedalWalker-v3")

## 8. Evaluate the trained policy

Loads the "latest" saved copy fresh from Drive (rather than reusing the
in-memory `model`/`vec_env`), so this cell also doubles as a sanity check
that what got saved actually works standalone.

In [ ]:
eval_vec_env = make_vec_env("BipedalWalker-v3", n_envs=1)
eval_vec_env = VecNormalize.load(os.path.join(DRIVE_DIR, "sac_bipedalwalker_vecnormalize_latest.pkl"), eval_vec_env)
eval_vec_env.training = False
eval_vec_env.norm_reward = False

eval_model = SAC.load(os.path.join(DRIVE_DIR, "sac_bipedalwalker_latest"), device="cpu")

mean_reward, std_reward = evaluate_policy(eval_model, eval_vec_env, n_eval_episodes=10, deterministic=True)
print(f"Trained SAC — mean reward: {mean_reward:.1f} +/- {std_reward:.1f}")
print(f"Random baseline — mean reward: {np.mean(random_rewards):.1f}")

## 9. Watch a rollout

In [ ]:
render_env = gym.make("BipedalWalker-v3", render_mode="rgb_array")
obs, info = render_env.reset(seed=0)

frames = []
terminated = truncated = False
total_reward = 0
while not (terminated or truncated):
    norm_obs = eval_vec_env.normalize_obs(obs)
    action, _ = eval_model.predict(norm_obs, deterministic=True)
    obs, reward, terminated, truncated, info = render_env.step(action)
    total_reward += reward
    frames.append(render_env.render())
render_env.close()

print(f"Rollout reward: {total_reward:.1f}, length: {len(frames)} steps")

n_show = 6
idxs = np.linspace(0, len(frames) - 1, n_show, dtype=int)
fig, axes = plt.subplots(1, n_show, figsize=(3 * n_show, 3))
for ax, i in zip(axes, idxs):
    ax.imshow(frames[i])
    ax.set_title(f"step {i}")
    ax.axis("off")
plt.tight_layout()
plt.show()

## Notes / things to tweak

- **Resuming**: set `CHECKPOINT_TO_RESUME` to a step count from the listing
  in section 2, rerun the notebook from section 1 down. `reset_num_timesteps=False`
  keeps the timestep counter (and therefore the learning-rate/entropy
  schedules, which are step-indexed) continuous across sessions instead of
  restarting them from zero.
- **`SAVE_EVERY_TIMESTEPS = 20_000`**: how often checkpoints get written. More
  frequent = less lost progress on a disconnect, at the cost of Drive storage
  and a brief pause each save (writing the replay buffer, especially, isn't
  free — it's up to `buffer_size` transitions).
- **Replay buffer file size**: with `buffer_size=300_000`, the saved replay
  buffer can be tens to low hundreds of MB depending on observation/action
  dims. Worth keeping an eye on Drive space if you checkpoint often; you can
  set `save_replay_buffer=False` in the callback if you're willing to accept
  a less-stable restart (empty buffer) in exchange for smaller checkpoints.
- **`gamma=0.98`, `train_freq=64`/`gradient_steps=64`, `net_arch=[400, 300]`,
  `log_std_init=-3`**: RL-Zoo-tuned settings for this env, addressing the
  "stand rigid and don't fall" local optimum where the agent's exploration
  noise collapses before it discovers a walking gait.
- **Symmetry augmentation** (not implemented here): BipedalWalker's two legs
  are mechanically identical, just mirrored — doubling the effective replay
  buffer via mirrored transitions is a natural next step if you want to push
  further without more compute.